# dbt Project Structure

## Overview
**dbt (data build tool)** is a transformation framework that lets analysts write SELECT statements and handles the surrounding infrastructure: materialisation, testing, documentation, lineage, and deployment.

**Core concepts:**

| Concept | What it is |
|---|---|
| **Model** | A `.sql` file containing a single SELECT statement; dbt creates a view or table from it |
| **Source** | A raw table loaded by an upstream process (ELT tool, Kafka, etc.) that dbt does not control |
| **Test** | A check run against a model or source after it builds (not null, unique, accepted values, etc.) |
| **Macro** | A reusable Jinja function, like a SQL helper function |
| **Seed** | A small CSV file dbt loads as a table (lookup tables, mappings) |
| **Snapshot** | SCD Type 2 history tracking for slowly changing dimensions |
| **Exposure** | Metadata declaring which dashboards/reports consume which models |

**dbt workflow:**
```
dbt run        → compile SQL + execute CREATE TABLE/VIEW statements
dbt test       → run all schema and singular tests
dbt docs generate + dbt docs serve  → build and browse the data lineage DAG
dbt build      → run + test in dependency order (recommended for CI)
```

**dbt is not:**
- A query runner for ad-hoc analysis
- An ingestion tool (no COPY, no API connectors)
- An orchestrator (dbt Cloud Scheduler / Airflow / Prefect handle scheduling)

---

In [1]:
# Display a realistic dbt project directory tree
import os

structure = [
    ("jaffle_shop/", "dbt project root"),
    ("  dbt_project.yml", "project config: name, version, model paths, materializations"),
    ("  profiles.yml", "connection credentials (usually ~/.dbt/profiles.yml)"),
    ("  packages.yml", "dbt Hub packages to install (dbt-utils, dbt-expectations, etc.)"),
    ("  seeds/", "small CSV lookup tables"),
    ("    payment_methods.csv", ""),
    ("  models/", "all SQL models live here"),
    ("    staging/", "one-to-one with source tables; light cleaning only"),
    ("      _sources.yml", "source declarations (database, schema, table names)"),
    ("      _staging.yml", "model descriptions and tests for this layer"),
    ("      stg_orders.sql", ""),
    ("      stg_customers.sql", ""),
    ("      stg_payments.sql", ""),
    ("    intermediate/", "joins, unions, business logic — not exposed to BI"),
    ("      int_order_payments_joined.sql", ""),
    ("    marts/", "final tables exposed to BI tools and analysts"),
    ("      finance/", ""),
    ("        fct_orders.sql", "fact table: one row per order"),
    ("        dim_customers.sql", "dimension table: one row per customer"),
    ("        _finance.yml", "descriptions and tests for mart models"),
    ("      core/", ""),
    ("        dim_dates.sql", ""),
    ("  macros/", "Jinja macros (reusable SQL snippets)"),
    ("    cents_to_dollars.sql", ""),
    ("    generate_schema_name.sql", ""),
    ("  tests/", "singular tests (custom SQL assertions)"),
    ("    assert_total_payment_positive.sql", ""),
    ("  analyses/", "ad-hoc SQL files tracked in version control but not run by dbt"),
    ("  snapshots/", "SCD Type 2 history models"),
    ("    customer_snapshot.sql", ""),
]

print("dbt project directory structure:")
print()
for path, note in structure:
    if note:
        print(f"  {path:<45s}  # {note}")
    else:
        print(f"  {path}")


dbt project directory structure:

  jaffle_shop/                                   # dbt project root
    dbt_project.yml                              # project config: name, version, model paths, materializations
    profiles.yml                                 # connection credentials (usually ~/.dbt/profiles.yml)
    packages.yml                                 # dbt Hub packages to install (dbt-utils, dbt-expectations, etc.)
    seeds/                                       # small CSV lookup tables
      payment_methods.csv
    models/                                      # all SQL models live here
      staging/                                   # one-to-one with source tables; light cleaning only
        _sources.yml                             # source declarations (database, schema, table names)
        _staging.yml                             # model descriptions and tests for this layer
        stg_orders.sql
        stg_customers.sql
        stg_payments.sql
      intermedia

---
## dbt_project.yml

In [2]:
dbt_project_yml = '''
name: jaffle_shop
version: '1.0.0'
config-version: 2

profile: jaffle_shop          # must match a profile in profiles.yml

model-paths: ["models"]
seed-paths:  ["seeds"]
test-paths:  ["tests"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

target-path: "target"         # compiled SQL and artifacts go here
clean-targets: ["target", "dbt_packages"]

# Default materializations per layer
models:
  jaffle_shop:
    staging:
      +materialized: view     # staging models are views (cheap to rebuild)
      +schema: staging        # written to the 'staging' schema

    intermediate:
      +materialized: ephemeral  # not written to the DB; inlined as CTEs

    marts:
      +materialized: table    # mart models are persistent tables
      +schema: marts

      finance:
        +materialized: table
        +tags: ["finance", "daily"]

vars:
  payment_methods: ['credit_card', 'coupon', 'bank_transfer', 'gift_card']
  start_date: '2018-01-01'
'''

print("dbt_project.yml:")
print(dbt_project_yml)


dbt_project.yml:

name: jaffle_shop
version: '1.0.0'
config-version: 2

profile: jaffle_shop          # must match a profile in profiles.yml

model-paths: ["models"]
seed-paths:  ["seeds"]
test-paths:  ["tests"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

target-path: "target"         # compiled SQL and artifacts go here
clean-targets: ["target", "dbt_packages"]

# Default materializations per layer
models:
  jaffle_shop:
    staging:
      +materialized: view     # staging models are views (cheap to rebuild)
      +schema: staging        # written to the 'staging' schema

    intermediate:
      +materialized: ephemeral  # not written to the DB; inlined as CTEs

    marts:
      +materialized: table    # mart models are persistent tables
      +schema: marts

      finance:
        +materialized: table
        +tags: ["finance", "daily"]

vars:
  payment_methods: ['credit_card', 'coupon', 'bank_transfer', 'gift_card']
  start_date: '2018-01-01'



---
## profiles.yml — connection configuration

In [3]:
profiles_yml = '''
# ~/.dbt/profiles.yml
# NEVER commit this file to version control — contains credentials

jaffle_shop:
  target: dev                   # default target when running locally

  outputs:
    dev:
      type: postgres
      host: localhost
      port: 5432
      user: "{{ env_var('DBT_USER') }}"
      password: "{{ env_var('DBT_PASSWORD') }}"
      dbname: analytics
      schema: dbt_dev_samantha  # personal dev schema — each analyst gets their own
      threads: 4                # parallel model execution

    prod:
      type: postgres
      host: prod-db.internal
      port: 5432
      user: dbt_prod
      password: "{{ env_var('DBT_PROD_PASSWORD') }}"
      dbname: analytics
      schema: analytics         # production schema
      threads: 8
'''

print("profiles.yml (stored at ~/.dbt/profiles.yml):")
print(profiles_yml)

print("Key points:")
notes = [
    "Each developer has a personal dev schema (dbt_dev_samantha) — no conflicts with teammates",
    "Credentials use env_var() — never hardcode passwords in YAML",
    "target: dev means 'dbt run' defaults to the dev target",
    "Override with: dbt run --target prod",
    "threads: N controls how many models run in parallel (up to N with no dependencies between them)",
]
for n in notes:
    print(f"  - {n}")


profiles.yml (stored at ~/.dbt/profiles.yml):

# ~/.dbt/profiles.yml
# NEVER commit this file to version control — contains credentials

jaffle_shop:
  target: dev                   # default target when running locally

  outputs:
    dev:
      type: postgres
      host: localhost
      port: 5432
      user: "{{ env_var('DBT_USER') }}"
      password: "{{ env_var('DBT_PASSWORD') }}"
      dbname: analytics
      schema: dbt_dev_samantha  # personal dev schema — each analyst gets their own
      threads: 4                # parallel model execution

    prod:
      type: postgres
      host: prod-db.internal
      port: 5432
      user: dbt_prod
      password: "{{ env_var('DBT_PROD_PASSWORD') }}"
      dbname: analytics
      schema: analytics         # production schema
      threads: 8

Key points:
  - Each developer has a personal dev schema (dbt_dev_samantha) — no conflicts with teammates
  - Credentials use env_var() — never hardcode passwords in YAML
  - target: dev means 'dbt 

---
## dbt CLI commands reference

In [4]:
print("=== dbt CLI commands ===")
commands = [
    ("dbt debug",                          "Test connection to the database"),
    ("dbt deps",                           "Install packages from packages.yml"),
    ("dbt seed",                           "Load CSV seeds into the database"),
    ("dbt run",                            "Build all models"),
    ("dbt run -s staging.*",               "Build only models in the staging directory"),
    ("dbt run -s +fct_orders",             "Build fct_orders and all its upstream dependencies"),
    ("dbt run -s fct_orders+",             "Build fct_orders and all downstream dependents"),
    ("dbt test",                           "Run all tests"),
    ("dbt test -s stg_orders",             "Run tests only for stg_orders"),
    ("dbt build",                          "Run + test in dependency order (recommended for CI)"),
    ("dbt docs generate",                  "Build documentation JSON artifacts"),
    ("dbt docs serve",                     "Launch local docs server at localhost:8080"),
    ("dbt compile -s fct_orders",          "Compile Jinja SQL to plain SQL without running"),
    ("dbt run --vars '{start_date: 2023-01-01}'", "Override a project variable at runtime"),
    ("dbt source freshness",               "Check if sources have been loaded recently"),
    ("dbt snapshot",                       "Run SCD Type 2 snapshot models"),
    ("dbt run --full-refresh",             "Drop and rebuild incremental/snapshot models"),
]
col = 45
print(f"  {'Command':<{col}}  Description")
print("  " + "-"*75)
for cmd, desc in commands:
    print(f"  {cmd:<{col}}  {desc}")


=== dbt CLI commands ===
  Command                                        Description
  ---------------------------------------------------------------------------
  dbt debug                                      Test connection to the database
  dbt deps                                       Install packages from packages.yml
  dbt seed                                       Load CSV seeds into the database
  dbt run                                        Build all models
  dbt run -s staging.*                           Build only models in the staging directory
  dbt run -s +fct_orders                         Build fct_orders and all its upstream dependencies
  dbt run -s fct_orders+                         Build fct_orders and all downstream dependents
  dbt test                                       Run all tests
  dbt test -s stg_orders                         Run tests only for stg_orders
  dbt build                                      Run + test in dependency order (recommended 

---
## Common Pitfalls

**1. Committing profiles.yml to version control**
`profiles.yml` typically lives at `~/.dbt/profiles.yml` (not inside the project repo) precisely because it contains credentials. If it ends up in a project folder that gets committed, database passwords are exposed in git history. Use `env_var()` for all credentials and add `profiles.yml` to `.gitignore` if it must be in the project directory.

**2. Running models directly in production from a local machine**
`dbt run --target prod` on a developer's laptop can overwrite production tables. Set `target: dev` as the default in profiles.yml and require explicit `--target prod` only in CI/CD pipelines with appropriate access controls.

**3. Naming models with the same name as source tables**
If a staging model is named `orders` and the source table is also `orders`, `ref('orders')` in downstream models is ambiguous and may point to the wrong object after a full refresh. The convention `stg_<source>_<table>` (e.g. `stg_stripe_payments`) avoids all naming collisions.

**4. Putting business logic in staging models**
Staging models should be one-to-one with source tables: rename columns, cast types, and handle NULL coalescing -- nothing more. Complex joins, business rules, and derived metrics belong in intermediate or mart models. Mixing concerns makes staging models fragile and hard to reuse.

**5. Not using +schema separators between layers**
Without `+schema` settings in `dbt_project.yml`, all models land in the same database schema. This makes it impossible to grant selective read access to analytics users (marts only) and impossible to differentiate staging data from production marts in monitoring.

**6. Forgetting --full-refresh when changing incremental model logic**
An incremental model's SELECT statement is only applied to new rows after the first build. If you change the column list, transformations, or filters in the model SQL, the existing rows in the table are not updated. Always run `dbt run --full-refresh -s <model>` after logic changes to incremental models.


---
*sql_methods_library - Samantha McGarrigle*